In [1]:
import tifffile as tiff
import rasterio
from rasterio.features import geometry_mask
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import sys
sys.path.append("..")  # Ajouter le dossier parent au chemin
from core2.eva_losses import eva_acc, eva_dice, eva_miou, eva_sensitivity, eva_specificity
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
#from eva_losses import eva_metrics

In [2]:
def get_metrics(ground_truth_img_path, segmentation_img_path, test_areas_path, show_images = False):

    ### Ouvrir image de Ground Truth
    with rasterio.open(ground_truth_img_path) as src:
        #print("Image ground truth:")
        #print(f"Dimensions (avant le masque)): {src.shape}") 
        #print(f"Projection : {src.crs}")
        ground_truth = src.read(1)
        src.close()

    #Importer le shapefile de la zone test
    test_areas = gpd.read_file(test_areas_path)
    #print(f"Projection des zones de test : {test_areas.crs}")

    # Créer un masque à partir des zones de test
    test_mask = geometry_mask(test_areas.geometry, transform=src.transform, invert=True, out_shape=ground_truth.shape)

    # Appliquer le masque à l'image de ground truth
    ground_truth = np.where(test_mask, ground_truth, np.nan)    

    # Vérifier qu'il y a seulement des 0 ou des 1 dans l'image
    ground_truth_nonan = ground_truth[~np.isnan(ground_truth)]
    print(f"Dimensions de l'image ground truth après application du masque : {ground_truth_nonan.shape}")
    unique_values_nonan = np.unique(ground_truth_nonan)
    if not np.array_equal(unique_values_nonan, [0, 1]):
        print(f"Erreur: l'image ground_truth contient d'autres valeurs (en ignorant les NaN) : {unique_values_nonan}")

    # Compter le nombre de pixels de chaque classe
    #unique, counts = np.unique(ground_truth_nonan, return_counts=True)
    #print("Nombre de pixels par classe :")
    #for value, count in zip(unique, counts):
    #    print(f"Classe {value}: {count} pixels")
    #print(f"proportion de pixels de la classe 1: {counts[1] / (counts[0] + counts[1]) * 100:.2f}%")

    #visualiser l'image de labels
    if show_images:
        plt.imshow(ground_truth, cmap="gray")
        plt.colorbar()
        plt.title("Image de ground_truth")
        plt.show()


    ### Ouvrir image de segmentation (output du modèle)r"C:\Users\sinadeau\OneDrive - NRCan RNCan\data_glo7030-projet\données_pour_metric\Données_JRW45_vrai_train_valid_test\10_epoch_train_valid_test\Fdc_JR_W45_2023_03_21_panIMG_0037_6_MS_alP1_cor2-001_inference_inference_seg.tif"
    with rasterio.open(segmentation_img_path) as src:
        #print("Image de segmentation")
        #print(f"Dimensions (avant le masque): {src.shape}")
        segmentation = src.read(1)
        src.close()

    # Créer un masque à partir des zones de test
    segmentation = np.where(test_mask, segmentation, np.nan)
    
    # Vérifier qu'il y a seulement des 0 ou des 1 dans l'image  
    segmentation_nonan = segmentation[~np.isnan(segmentation)]
    print(f"Dimensions de l'image de segmentation après application du masque : {segmentation_nonan.shape}")
    unique_values_nonan = np.unique(segmentation_nonan)
    if not np.array_equal(unique_values_nonan, [0, 1]):
        print(f"Erreur: l'image de segmentation contient d'autres valeurs (en ignorant les NaN) : {unique_values_nonan}")

    # Compter le nombre de pixels de chaque classe
    #unique, counts = np.unique(segmentation_nonan, return_counts=True)
    #print("Nombre de pixels par classe :")
    #for value, count in zip(unique, counts):
    #    print(f"Classe {value}: {count} pixels")
    #print(f"proportion de pixels de la classe 1: {counts[1] / (counts[0] + counts[1]) * 100:.2f}%")

    #visualiser l'image de segmentation
    if show_images:
        plt.imshow(segmentation, cmap="gray")
        plt.colorbar()
        plt.title("Image de segmentation")
        plt.show()

    ### Calculer les métriques
    eva_acc_value = eva_acc(ground_truth_nonan, segmentation_nonan)
    #eva_dice_value = eva_dice(ground_truth_nonan, segmentation_nonan) La même chose que le f1 score
    eva_miou_value = eva_miou(ground_truth_nonan, segmentation_nonan)
    eva_sensitivity_value = eva_sensitivity(ground_truth_nonan, segmentation_nonan)
    eva_specificity_value = eva_specificity(ground_truth_nonan, segmentation_nonan)

    #fonction de scikit-learn pour calculer les métriques
    #acc = accuracy_score(ground_truth_nonan.flatten(), segmentation_nonan.flatten())
    f1 = f1_score(ground_truth_nonan.flatten(), segmentation_nonan.flatten())
    precision = precision_score(ground_truth_nonan.flatten(), segmentation_nonan.flatten())
    recall = recall_score(ground_truth_nonan.flatten(), segmentation_nonan.flatten())
    roc_auc_score_value = roc_auc_score(ground_truth_nonan.flatten(), segmentation_nonan.flatten())  
    cm = confusion_matrix(ground_truth_nonan.flatten(), segmentation_nonan.flatten())    

    return eva_acc_value, f1, precision, recall, eva_miou_value, eva_sensitivity_value, eva_specificity_value, roc_auc_score_value, cm


## JRW45

In [3]:
ground_truth_img_path= r"C:\Users\sinadeau\OneDrive - NRCan RNCan\data_glo7030-projet\données_pour_metric\Données_JRW45_vrai_train_valid_test\10_epoch_train_valid_test\gt_all.png"
segmentation_img_path = r"C:\Users\sinadeau\OneDrive - NRCan RNCan\data_glo7030-projet\données_pour_metric\Données_JRW45_vrai_train_valid_test\10_epoch_train_valid_test\Fdc_JR_W45_2023_03_21_panIMG_0037_6_MS_alP1_cor2-001_inference_inference_seg.tif"

#Métriques en train
print("train")
train_areas_path = r'C:\Users\sinadeau\OneDrive - NRCan RNCan\data_glo7030-projet\Fdc_JR_W45\train_test_split\train_areas.shp'
metrics_train = get_metrics(ground_truth_img_path, segmentation_img_path, train_areas_path)

#Métriques en val
print("Val")
val_areas_path = r'C:\Users\sinadeau\OneDrive - NRCan RNCan\data_glo7030-projet\Fdc_JR_W45\train_test_split\val_areas.shp'
metrics_val = get_metrics(ground_truth_img_path, segmentation_img_path, val_areas_path)

#Métriques en test
#print("Test")
#test_areas_path = r'C:\Users\sinadeau\OneDrive - NRCan RNCan\data_glo7030-projet\Fdc_JR_W45\train_test_split\test_areas.shp'
#metrics_test = get_metrics(ground_truth_img_path, segmentation_img_path, test_areas_path)

metrics_combined = pd.DataFrame({
    "Train": metrics_train[:-1],
    "Validation": metrics_val[:-1]
}, index=["acc", "f1", "precision", "recall", "miou", "sensitivity", "specificity", "roc_auc"])

print(metrics_combined)

print(metrics_val[-1])

train
Dimensions de l'image ground truth après application du masque : (11891931,)
Dimensions de l'image de segmentation après application du masque : (11891931,)
Val
Dimensions de l'image ground truth après application du masque : (1961181,)
Dimensions de l'image de segmentation après application du masque : (1961181,)
                Train  Validation
acc          0.959434    0.947963
f1           0.913240    0.884891
precision    0.936756    0.946841
recall       0.890876    0.830550
miou         0.894377    0.864250
sensitivity  0.890876    0.830550
specificity  0.981042    0.985208
roc_auc      0.935959    0.907879
[[1466865   22023]
 [  80030  392263]]
